In [3]:
from predict_loop_functions import predict_loop, visual_prediction
import sys
import os
sys.path.append('C://Users//joshf//OneDrive//GitHub//fnn-haefner//fnn//microns')
from __init__ import scan
import numpy as np
from numpy import full, concatenate
import pandas as pd

In [4]:
test_image = np.full(shape = (5, 144, 256), fill_value = 128)


In [ ]:
test_sum, test_mean, test_var = predict_loop("dynamic", test_image, 3, [[4,7]], True, noise_seeds = 5, num_frames = 3)

In [6]:
neuron = 7167

In [7]:
test_var[:, neuron]

array([[1.79285024e-03, 9.00000000e+00],
       [7.96219124e-03, 9.00000000e+00],
       [2.82287767e-03, 9.00000000e+00],
       [5.19111923e-03, 9.00000000e+00],
       [6.43901423e-03, 9.00000000e+00]])

In [8]:
test_mean[:, neuron]

array([[3.00256804, 9.        ],
       [3.02460082, 9.        ],
       [3.03190856, 9.        ],
       [3.00569701, 9.        ],
       [3.07571727, 9.        ]])

In [9]:
test_sum[:, :, neuron]

array([[[3.06127983, 9.        ],
        [3.00460404, 9.        ],
        [3.00380844, 9.        ],
        [2.92920518, 9.        ],
        [3.01394272, 9.        ]],

       [[3.12333864, 9.        ],
        [2.95836484, 9.        ],
        [2.92747295, 9.        ],
        [3.14103431, 9.        ],
        [2.97279334, 9.        ]],

       [[3.02940911, 9.        ],
        [3.0955326 , 9.        ],
        [2.94619656, 9.        ],
        [3.07892478, 9.        ],
        [3.00947976, 9.        ]],

       [[3.13058573, 9.        ],
        [3.02102935, 9.        ],
        [2.92160243, 9.        ],
        [2.95046455, 9.        ],
        [3.004803  , 9.        ]],

       [[3.04539734, 9.        ],
        [2.9620356 , 9.        ],
        [3.20811439, 9.        ],
        [3.0620783 , 9.        ],
        [3.10096073, 9.        ]]])

In [5]:
test_sum[0][1].shape

(7493,)

#### EDA microns_area_labels and import area labels

In [6]:
brain_region_df = pd.read_csv('brain_region_files//microns_area_labels.csv')

In [7]:
brain_region_df.head()

,session,scan_idx,unit_id,brain_area
0,4,7,1,LM
1,4,7,2,LM
2,4,7,3,LM
3,4,7,4,LM
4,4,7,5,LM


In [8]:
scans_filter_df = brain_region_df[((brain_region_df['session'] == 5) & (brain_region_df['scan_idx'] == 7))
                                  ^ (brain_region_df['session'] == 4) & (brain_region_df['scan_idx'] == 7)]

In [9]:
scans_filter_df.head()

,session,scan_idx,unit_id,brain_area
0,4,7,1,LM
1,4,7,2,LM
2,4,7,3,LM
3,4,7,4,LM
4,4,7,5,LM


In [10]:
len(scans_filter_df['unit_id'])

17257

In [11]:
scans_filter_df.shape

(17257, 4)

In [12]:
model, table = scan(5, 7)

In [13]:
table.head()

,session,scan_idx,unit_id
readout_id,,,
0,5,7,2
1,5,7,3
2,5,7,4
3,5,7,6
4,5,7,7


In [14]:
table.shape

(8138, 3)

#### join dataframes for one scan and encode

In [15]:
# join dataframes on equivalent values for session and scan 
join_df = pd.merge(table, scans_filter_df, how = 'inner', on = ['session', 'scan_idx', 'unit_id'])

In [16]:
join_df.shape

(8138, 4)

In [17]:
join_df.head()

,session,scan_idx,unit_id,brain_area
0,5,7,2,LM
1,5,7,3,LM
2,5,7,4,LM
3,5,7,6,LM
4,5,7,7,LM


In [18]:
join_df['brain_area'].value_counts()

brain_area
V1    5777
LM    1066
RL     892
AL     403
Name: count, dtype: int64

In [19]:
# encode as follows -> 1:V1 , 2:LM , 3:AL , 4:RL

encoding = {'V1':1, 'LM':2, 'AL':3, 'RL':4}
join_df['brain_area'] = join_df['brain_area'].map(encoding)

In [20]:
join_df.head()

,session,scan_idx,unit_id,brain_area
0,5,7,2,2
1,5,7,3,2
2,5,7,4,2
3,5,7,6,2
4,5,7,7,2


#### join for two scans and encode

In [21]:
stimuli = np.full(shape = (10, 144, 256), fill_value = 128).astype('uint8')

In [22]:
def visual_prediction(session: int, scan_idx: int, stimuli_noise) -> np.array:
    """
    Parameters 
    ----------
    session: int
        session number 
    scan_idx: int
        scan id
    stimuli_noise: array
        array/dataframe with added noise    

    Returns
    -------
    array
        neuron predictions
    array
        ids with corresponding unit_id for each scan
    """
    pred_model, ids = scan(session, scan_idx, directory = os.path.join(os.getcwd(), "data","microns")) # look at data/microns/scans.csv for numbers that work
    results = pred_model.predict(stimuli = stimuli_noise)
    return results , ids  

In [23]:
"""
in code, results1 is one element in an array and results2 is another, then said array is stacked
"""
# prediction1_array = [visual_prediction(pair[0], pair[1], transformed_image)[0] for pair in scans]

test = [visual_prediction(4, 7, stimuli), visual_prediction(5, 7, stimuli)]

In [24]:
test_predictions = [output[0] for output in test]
test_ids = [output[1] for output in test]

In [25]:
print(f'shapes for following: predictions[0]: {test_predictions[0].shape} ; predictions[1]: {test_predictions[1].shape}\n'
      f'ids[0]: {test_ids[0].shape} ; ids[1]: {test_ids[1].shape}')

shapes for following: predictions[0]: (10, 7493) ; predictions[1]: (10, 8138)
ids[0]: (7493, 3) ; ids[1]: (8138, 3)


In [26]:
test_predictions = np.concatenate(test_predictions, axis = 1)
print(f'test_predictions new shape {test_predictions.shape}')

test_predictions new shape (10, 15631)


In [27]:
test_ids = np.concatenate(test_ids, axis = 0) # add on axis 0 here, easier to visualize
print(f'test_ids new shape {test_ids.shape}')

test_ids new shape (15631, 3)


In [28]:
test_ids[7492]

array([   4,    7, 8395])

In [29]:
test_predictions.shape

(10, 15631)

In [30]:
brain_region_df.head()

,session,scan_idx,unit_id,brain_area
0,4,7,1,LM
1,4,7,2,LM
2,4,7,3,LM
3,4,7,4,LM
4,4,7,5,LM


In [31]:
test_ids_df = pd.DataFrame(test_ids, columns = ['session', 'scan_idx', 'unit_id'])
test_ids_df.head()

,session,scan_idx,unit_id
0,4,7,1
1,4,7,3
2,4,7,4
3,4,7,5
4,4,7,6


In [32]:
# match test_ids to brain_region_df
# encode brain_area

final_match_id = pd.merge(brain_region_df, test_ids_df, how = 'inner', on = ['session', 'scan_idx', 'unit_id'])

In [33]:
final_match_id.shape

(15631, 4)

In [34]:
encoding = {'V1':1, 'LM':2, 'AL':3, 'RL':4}

final_match_id['brain_area'] = final_match_id['brain_area'].map(encoding)

In [35]:
final_match_id.head()

,session,scan_idx,unit_id,brain_area
0,4,7,1,2
1,4,7,3,2
2,4,7,4,2
3,4,7,5,2
4,4,7,6,2


In [36]:
print(f'test_predictions shape {test_predictions.shape} ; final_match_id shape {final_match_id.shape}')

test_predictions shape (10, 15631) ; final_match_id shape (15631, 4)


here we only want to add the LAST COLUMN for final match id to test predictions

In [37]:
test_predictions.shape

(10, 15631)

In [38]:
length = final_match_id.shape[0]
final_brain_area = final_match_id['brain_area'].to_numpy().reshape(1, length, 1)
final_brain_area = np.broadcast_to(final_brain_area, shape = (test_predictions.shape[0] , test_predictions.shape[1], 1))

In [39]:
final_brain_area.shape

(10, 15631, 1)

In [40]:
test_predictions = test_predictions[:,:,np.newaxis]

In [41]:
test_predictions.shape

(10, 15631, 1)

In [42]:
final_array = np.concatenate([test_predictions, final_brain_area], axis=2)

In [43]:
final_array.shape

(10, 15631, 2)

#### final_function 1

In [ ]:
# import brain region ONCE somewhere in code
brain_regions_df = pd.read_csv('brain_region_files//microns_area_labels.csv')

def add_brain_region(predictions_ids: list, brain_regions: pd.DataFrame) -> np.array:
    """
    Parameters
    ----------
    predictions_ids: list
        list of array objects, each containing prediction and id for relevant neurons
    brain_regions: DataFrame
        brain region mappings from csv file

    Returns
    array
        array of predictions with brain region column added
    """
    predictions = [output[0] for output in predictions_ids]
    ids = [output[1] for output in predictions_ids]

    predictions = np.concatenate(predictions, axis = 1)
    ids = np.concatenate(ids, axis = 0)

    # make dataframe to merge with brain region data from csv
    ids_df = pd.DataFrame(ids, columns = ['session', 'scan_idx', 'unit_id'])
    
    ids_matched = pd.merge(brain_regions, ids_df, how = 'inner', on = ['session', 'scan_idx', 'unit_id'])

    encoding = {'V1':1, 'LM':2, 'AL':3, 'RL':4}

    ids_matched['brain_area'] = ids_matched['brain_area'].map(encoding)

    length = ids_matched.shape[0]

    ids_matched_array = ids_matched['brain_area'].to_numpy().reshape(1, length, 1)

    predictions = predictions[:, :, np.newaxis]

    return np.concatenate([predictions, ids_matched_array], axis = 2)

#### final function 2

In [5]:
brain_regions_df = pd.read_csv('brain_region_files//microns_area_labels.csv')

In [21]:
def add_brain_region(predictions_ids: list, brain_regions: pd.DataFrame, encoding = {'V1':1, 'LM':2, 'AL':3, 'RL':4}) -> np.array:
    """
    Parameters
    ----------
    predictions_ids: list
        list of array objects, each containing prediction and id for relevant neurons
    brain_regions: DataFrame
        brain region mappings from csv file
    encoding: dictionary
        default dictionary provided for brain regions V1, LM, AL and RL

    Returns
    array
        array of predictions with brain region column added
    """
    predictions = np.concatenate([output[0] for output in predictions_ids], axis = 1)
    ids = np.concatenate([output[1] for output in predictions_ids], axis = 0)

    # make dataframe to merge with brain region data from csv
    ids_df = pd.DataFrame(ids, columns = ['session', 'scan_idx', 'unit_id'])
    
    ids_matched = pd.merge(brain_regions, ids_df, how = 'inner', on = ['session', 'scan_idx', 'unit_id'])['brain_area'] #only need brain_area

    predictions = predictions[:, :, np.newaxis]

    ids_matched = ids_matched.map(encoding).to_numpy()
    ids_matched = ids_matched[np.newaxis, :, np.newaxis]
    ids_matched = np.broadcast_to(ids_matched, shape = (predictions.shape[0], predictions.shape[1], 1))

    return np.concatenate([predictions, ids_matched], axis = 2)

In [7]:
def visual_prediction(session: int, scan_idx: int, stimuli_noise) -> np.array:
    """
    Parameters 
    ----------
    session: int
        session number 
    scan_idx: int
        scan id
    stimuli_noise: array
        array/dataframe with added noise    

    Returns
    -------
    array
        neuron predictions
    array
        ids with corresponding unit_id for each scan
    """
    pred_model, ids = scan(session, scan_idx, directory = os.path.join(os.getcwd(), "data","microns")) # look at data/microns/scans.csv for numbers that work
    results = pred_model.predict(stimuli = stimuli_noise)
    return results , ids  

In [25]:
results1 = visual_prediction(4, 7, test_image.astype('uint8'))
results2 = visual_prediction(5, 7, test_image.astype('uint8'))

In [26]:
final_test = add_brain_region([results1, results2], brain_regions_df)

In [27]:
final_test.shape

(5, 15631, 2)

In [35]:
final_test[0][10000]

array([0.12878375, 2.        ])